In [2]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt

# ==================================================
# CONNECT DB
# ==================================================

con = duckdb.connect(
    "../data/warehouse/ecommerce.duckdb",
    read_only=True
)

In [6]:
# ==================================================
# INSIGHT 1
# FUNNEL ANALYSIS
# ==================================================

print("\n")
print("=" * 80)
print("INSIGHT 1: CUSTOMER FUNNEL LEAKAGE")
print("=" * 80)

df_funnel = con.execute("""
SELECT *
FROM mart_funnel
ORDER BY stage_name
""").df()

print(df_funnel)

view_users = df_funnel.loc[
    df_funnel["stage_name"] == "View",
    "user_count"
].iloc[0]

cart_users = df_funnel.loc[
    df_funnel["stage_name"] == "Cart",
    "user_count"
].iloc[0]

purchase_users = df_funnel.loc[
    df_funnel["stage_name"] == "Purchase",
    "user_count"
].iloc[0]

view_to_cart = cart_users / view_users
cart_to_purchase = purchase_users / cart_users
overall_cr = purchase_users / view_users

print(f"""
[FINDING]

Users Viewed     : {view_users:,.0f}
Users Carted     : {cart_users:,.0f}
Users Purchased  : {purchase_users:,.0f}

View → Cart CR       : {view_to_cart:.2%}
Cart → Purchase CR   : {cart_to_purchase:.2%}
Overall CR           : {overall_cr:.2%}

[BUSINESS IMPACT]

Traffic đang rất lớn nhưng một lượng lớn user
rời khỏi hành trình trước khi hoàn tất mua hàng.

[RECOMMENDATION]

1. Tối ưu PDP (Product Detail Page)
2. Tối ưu Checkout Flow
3. Remarketing cho nhóm Cart Abandonment
""")



INSIGHT 1: CUSTOMER FUNNEL LEAKAGE
  stage_name  user_count  conversion_rate
0       Cart      826317           0.2236
1   Purchase      441638           0.1195
2       View     3695598           1.0000

[FINDING]

Users Viewed     : 3,695,598
Users Carted     : 826,317
Users Purchased  : 441,638

View → Cart CR       : 22.36%
Cart → Purchase CR   : 53.45%
Overall CR           : 11.95%

[BUSINESS IMPACT]

Traffic đang rất lớn nhưng một lượng lớn user
rời khỏi hành trình trước khi hoàn tất mua hàng.

[RECOMMENDATION]

1. Tối ưu PDP (Product Detail Page)
2. Tối ưu Checkout Flow
3. Remarketing cho nhóm Cart Abandonment



In [7]:
# ==================================================
# INSIGHT 2
# RETENTION
# ==================================================

print("\n")
print("=" * 80)
print("INSIGHT 2: CUSTOMER RETENTION")
print("=" * 80)

df_retention = con.execute("""
SELECT
    cohort_index,
    AVG(retention_rate) as retention_rate
FROM mart_retention
GROUP BY 1
ORDER BY 1
""").df()

print(df_retention)

week0 = df_retention.iloc[0]["retention_rate"]

week_last = df_retention.iloc[
    len(df_retention)-1
]["retention_rate"]

print(f"""
[FINDING]

Week 0 Retention : {week0:.2%}
Last Week Retention : {week_last:.2%}

[BUSINESS IMPACT]

Khách hàng mới không quay lại đủ nhiều.

Doanh thu hiện tại đang phụ thuộc mạnh
vào việc liên tục mua traffic mới.

[RECOMMENDATION]

1. Loyalty Program
2. Retention Campaign
3. Personalized Offers
4. Email Remarketing
""")



INSIGHT 2: CUSTOMER RETENTION
   cohort_index  retention_rate
0             0         1.00000
1             1         0.30975
2             2         0.29150
3             3         0.26165
4             4         0.27320

[FINDING]

Week 0 Retention : 100.00%
Last Week Retention : 27.32%

[BUSINESS IMPACT]

Khách hàng mới không quay lại đủ nhiều.

Doanh thu hiện tại đang phụ thuộc mạnh
vào việc liên tục mua traffic mới.

[RECOMMENDATION]

1. Loyalty Program
2. Retention Campaign
3. Personalized Offers
4. Email Remarketing



In [8]:
# ==================================================
# INSIGHT 3
# CUSTOMER SEGMENTS
# ==================================================

print("\n")
print("=" * 80)
print("INSIGHT 3: RFM CUSTOMER SEGMENTS")
print("=" * 80)

df_segment = con.execute("""
SELECT
    customer_segment,
    COUNT(*) as users,
    SUM(monetary) as revenue
FROM mart_customer_segments
GROUP BY 1
ORDER BY revenue DESC
""").df()

print(df_segment)

top_segment = df_segment.iloc[0]

print(f"""
[FINDING]

Top Revenue Segment:

{top_segment['customer_segment']}

Revenue:

{top_segment['revenue']:,.2f}

[BUSINESS IMPACT]

Một nhóm khách hàng nhỏ tạo ra
phần lớn doanh thu.

[RECOMMENDATION]

1. VIP Program
2. Exclusive Promotion
3. Personalized Experience
4. Churn Prevention Strategy
""")



INSIGHT 3: RFM CUSTOMER SEGMENTS
  customer_segment   users       revenue
0   Lost Customers   62833  1.194907e+08
1  Needs Attention  119135  6.340328e+07
2    New Customers   37453  3.724897e+07
3   About To Sleep   27765  2.907472e+07
4  Loyal Customers  100490  1.596994e+07
5          At Risk   50103  6.315654e+06
6        Champions   43859  3.689855e+06

[FINDING]

Top Revenue Segment:

Lost Customers

Revenue:

119,490,651.87

[BUSINESS IMPACT]

Một nhóm khách hàng nhỏ tạo ra
phần lớn doanh thu.

[RECOMMENDATION]

1. VIP Program
2. Exclusive Promotion
3. Personalized Experience
4. Churn Prevention Strategy



In [9]:
# ==================================================
# INSIGHT 4
# CATEGORY PERFORMANCE
# ==================================================

print("\n")
print("=" * 80)
print("INSIGHT 4: CATEGORY PERFORMANCE")
print("=" * 80)

df_category = con.execute("""
SELECT
    category_group,
    SUM(total_sessions) as sessions,
    SUM(cart_sessions) as carts,
    SUM(purchase_sessions) as purchases
FROM int_daily_category_metrics
GROUP BY 1
ORDER BY purchases DESC
""").df()

print(df_category)

print("""
[FINDING]

Một số category thu hút lượng lớn traffic
nhưng chuyển đổi thấp.

[BUSINESS IMPACT]

Traffic không đồng nghĩa với doanh thu.

Cần tập trung tối ưu nhóm category
có volume cao nhưng CR thấp.

[RECOMMENDATION]

1. Category-specific Promotion
2. Improve Product Assortment
3. Optimize Product Ranking
""")



INSIGHT 4: CATEGORY PERFORMANCE
   category_group   sessions     carts  purchases
0     electronics  5990736.0  887359.0   417810.0
1                  5108374.0  503945.0   208628.0
2      appliances  1643443.0  206156.0    88390.0
3       computers   933115.0   75073.0    30140.0
4         apparel   697283.0   33984.0    12659.0
5       furniture   523729.0   26990.0    10333.0
6            auto   307529.0   24618.0    10054.0
7    construction   251337.0   21471.0     7998.0
8            kids   262848.0   13359.0     5598.0
9     accessories   103340.0    5120.0     2019.0
10          sport    72296.0    3791.0     1368.0
11       medicine     6229.0     798.0      330.0
12     stationery     5192.0     423.0      159.0
13   country_yard     5884.0     220.0       60.0

[FINDING]

Một số category thu hút lượng lớn traffic
nhưng chuyển đổi thấp.

[BUSINESS IMPACT]

Traffic không đồng nghĩa với doanh thu.

Cần tập trung tối ưu nhóm category
có volume cao nhưng CR thấp.

[RECOMMENDATI

In [11]:
# ==================================================
# INSIGHT 5
# CAUSAL ANALYSIS
# ==================================================

print("\n")
print("=" * 80)
print("INSIGHT 5: DIFFERENCE-IN-DIFFERENCES FRAMEWORK")
print("=" * 80)

df_did = con.execute("""
SELECT
    is_treatment,
    is_post,
    AVG(cart_to_purchase_cr) as avg_cr
FROM mart_causal_did
GROUP BY 1,2
ORDER BY 1,2
""").df()

print(df_did)

# --------------------------------------------------
# Check đủ 4 nhóm hay chưa
# --------------------------------------------------

required_groups = [
    (0, 0),  # Control Pre
    (0, 1),  # Control Post
    (1, 0),  # Treatment Pre
    (1, 1)   # Treatment Post
]

available_groups = set(
    zip(
        df_did["is_treatment"],
        df_did["is_post"]
    )
)

if all(group in available_groups for group in required_groups):

    control_pre = df_did[
        (df_did["is_treatment"] == 0)
        & (df_did["is_post"] == 0)
    ]["avg_cr"].iloc[0]

    control_post = df_did[
        (df_did["is_treatment"] == 0)
        & (df_did["is_post"] == 1)
    ]["avg_cr"].iloc[0]

    treatment_pre = df_did[
        (df_did["is_treatment"] == 1)
        & (df_did["is_post"] == 0)
    ]["avg_cr"].iloc[0]

    treatment_post = df_did[
        (df_did["is_treatment"] == 1)
        & (df_did["is_post"] == 1)
    ]["avg_cr"].iloc[0]

    did_effect = (
        (treatment_post - treatment_pre)
        -
        (control_post - control_pre)
    )

    print(f"""
[FINDING]

Treatment Pre  : {treatment_pre:.4f}
Treatment Post : {treatment_post:.4f}

Control Pre    : {control_pre:.4f}
Control Post   : {control_post:.4f}

Estimated DID Effect : {did_effect:.4f}

[BUSINESS IMPACT]

Ước lượng tác động thuần của thay đổi chính sách
lên tỷ lệ chuyển đổi sau khi loại bỏ ảnh hưởng
của xu hướng thị trường chung.

[RECOMMENDATION]

1. Mở rộng thử nghiệm trên nhiều category hơn
2. Theo dõi thêm Revenue Lift
3. Kết hợp A/B Testing để xác nhận kết quả
""")

else:

    print("""
[FINDING]

Dataset hiện tại không chứa đầy đủ 4 nhóm:

- Control Pre
- Control Post
- Treatment Pre
- Treatment Post

nên không thể tính Difference-in-Differences
một cách thống kê đáng tin cậy.

[BUSINESS IMPACT]

Đây là hạn chế của dữ liệu hiện có,
không phải lỗi của mô hình phân tích.

[RECOMMENDATION]

Để triển khai Causal Analysis thực tế cần:

1. Xác định rõ Treatment Group
2. Xác định rõ Control Group
3. Thu thập dữ liệu trước thay đổi (Pre)
4. Thu thập dữ liệu sau thay đổi (Post)

Framework mart_causal_did đã được xây dựng
để sẵn sàng áp dụng cho các thử nghiệm tương lai.
""")



INSIGHT 5: DIFFERENCE-IN-DIFFERENCES FRAMEWORK
   is_treatment  is_post    avg_cr
0             1        0  0.784127
1             1        1  0.514127

[FINDING]

Dataset hiện tại không chứa đầy đủ 4 nhóm:

- Control Pre
- Control Post
- Treatment Pre
- Treatment Post

nên không thể tính Difference-in-Differences
một cách thống kê đáng tin cậy.

[BUSINESS IMPACT]

Đây là hạn chế của dữ liệu hiện có,
không phải lỗi của mô hình phân tích.

[RECOMMENDATION]

Để triển khai Causal Analysis thực tế cần:

1. Xác định rõ Treatment Group
2. Xác định rõ Control Group
3. Thu thập dữ liệu trước thay đổi (Pre)
4. Thu thập dữ liệu sau thay đổi (Post)

Framework mart_causal_did đã được xây dựng
để sẵn sàng áp dụng cho các thử nghiệm tương lai.

